In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

LABEL_COLUMNS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

class AdvancedBayesianRefiner:
    def __init__(self, interaction_threshold=0.1, C_reg=0.1):
        self.interaction_threshold = interaction_threshold
        self.C_reg = C_reg
        self.models = {}
        self.scaler = StandardScaler()
        self.significant_pairs = []
        
    def _create_features(self, ens_preds, text_preds):
        features = []
        features.append(ens_preds)
        features.append(text_preds)
        
        for i, j in self.significant_pairs:
            interaction = ens_preds[:, i] * ens_preds[:, j]
            features.append(interaction.reshape(-1, 1))
                
        return np.hstack(features)

    def fit(self, gt_df, text_df, ens_df):
        print("🔄 Подготовка данных...")
        
        # 1. Находим строки с реальными GT лейблами (не NaN)
        # Проверяем, есть ли хотя бы один не-NaN лейбл в строке
        mask_gt_exists = gt_df[LABEL_COLUMNS].notna().any(axis=1)
        gt_labeled_df = gt_df[mask_gt_exists].copy()
        
        print(f"   Найдено {len(gt_labeled_df)} строк с GT лейблами.")
        
        if len(gt_labeled_df) == 0:
            raise ValueError("Нет строк с GT лейблами!")

        # 2. Выравниваем все три датафрейма по этим UID
        labeled_uids = gt_labeled_df['StudyInstanceUID'].values
        
        # Берем соответствующие строки из text_df и ens_df
        text_labeled = text_df[text_df['StudyInstanceUID'].isin(labeled_uids)].set_index('StudyInstanceUID').loc[labeled_uids].reset_index()
        ens_labeled = ens_df[ens_df['StudyInstanceUID'].isin(labeled_uids)].set_index('StudyInstanceUID').loc[labeled_uids].reset_index()
        gt_labeled = gt_labeled_df.set_index('StudyInstanceUID').loc[labeled_uids].reset_index()

        # 3. Извлекаем массивы
        # Важно: в gt_labeled могут быть NaN для некоторых лейблов. 
        # Мы будем обучать каждую модель только на тех строках, где этот конкретный лейбл не NaN.
        ens_preds_all = ens_labeled[LABEL_COLUMNS].values.astype(float)
        text_preds_all = text_labeled[LABEL_COLUMNS].values.astype(float)
        
        # 4. Определяем значимые пары взаимодействий
        # Считаем корреляции на ВСЕХ данных ансамбля (4407 строк), так как они отражают общие зависимости
        # Это более надежно, чем считать на 58 строках
        ens_all_preds = ens_df[LABEL_COLUMNS].values.astype(float)
        # Заменяем NaN на 0.5 для расчета корреляций
        ens_all_preds_clean = np.nan_to_num(ens_all_preds, nan=0.5)
        
        corr_matrix = np.corrcoef(ens_all_preds_clean.T)
        self.significant_pairs = []
        
        for i in range(len(LABEL_COLUMNS)):
            for j in range(i+1, len(LABEL_COLUMNS)):
                if abs(corr_matrix[i, j]) > self.interaction_threshold:
                    self.significant_pairs.append((i, j))
                    
        print(f"   Найдено {len(self.significant_pairs)} значимых пар взаимодействий (на основе всего ансамбля).")

        # 5. Обучение моделей для каждого лейбла
        for idx, col in enumerate(LABEL_COLUMNS):
            # Берем GT только для текущего лейбла
            y_train_full = gt_labeled[col].values.astype(float)
            
            # Маска: где этот конкретный лейбл не NaN
            mask_valid_label = ~np.isnan(y_train_full)
            
            y_train = y_train_full[mask_valid_label]
            ens_train = ens_preds_all[mask_valid_label]
            text_train = text_preds_all[mask_valid_label]
            
            if len(y_train) < 5:
                print(f"   ⚠️ Пропуск {col}: слишком мало размеченных данных ({len(y_train)})")
                continue
                
            if len(np.unique(y_train)) < 2:
                print(f"   ⚠️ Пропуск {col}: нет вариации в GT")
                continue

            # Создаем признаки только для размеченных строк
            X_train = self._create_features(ens_train, text_train)
            X_train = np.nan_to_num(X_train, nan=0.5)
            
            # Масштабируем
            X_train_scaled = self.scaler.fit_transform(X_train)
            
            model = LogisticRegression(
                penalty='l1', 
                solver='liblinear', 
                C=self.C_reg, 
                max_iter=1000,
                class_weight='balanced'
            )
            
            try:
                model.fit(X_train_scaled, y_train)
                self.models[col] = model
                print(f"   ✅ Модель для {col} обучена на {len(y_train)} примерах.")
            except Exception as e:
                print(f"   ❌ Ошибка при обучении {col}: {e}")
            
        print("✅ Все модели обучены.")
        return self
    
    def transform(self, text_df, ens_df):
        print("🔄 Переоценка предсказаний...")
        
        ens_preds = ens_df[LABEL_COLUMNS].values.astype(float)
        text_preds = text_df[LABEL_COLUMNS].values.astype(float)
        
        X_test = self._create_features(ens_preds, text_preds)
        X_test = np.nan_to_num(X_test, nan=0.5)
        X_test_scaled = self.scaler.transform(X_test)
        
        refined_probs = np.zeros_like(ens_preds)
        
        for idx, col in enumerate(LABEL_COLUMNS):
            if col in self.models:
                proba = self.models[col].predict_proba(X_test_scaled)[:, 1]
                refined_probs[:, idx] = proba
            else:
                refined_probs[:, idx] = ens_preds[:, idx]
                
        result_df = ens_df[['StudyInstanceUID']].copy()
        result_df[LABEL_COLUMNS] = refined_probs
        
        print("✅ Переоценка завершена.")
        return result_df


# ============================================================
# ИСПОЛЬЗОВАНИЕ
# ============================================================

gt_df = pd.read_csv("data/train.csv")       
text_df = pd.read_csv("data/consensus_labels.csv") 
df6 = pd.read_csv("data/oof.csv") 

refiner = AdvancedBayesianRefiner(interaction_threshold=0.1, C_reg=0.3)
refiner.fit(gt_df, text_df, df6)

refined_df = refiner.transform(text_df, df6)
refined_df.to_csv("labels/refined_predictions_v1.csv", index=False)